# Recs 008: History-blend grid search (@5 metrics)

## Key Goal

Quantify whether history-aware query blending improves retrieval over raw query embedding using a same-user proxy task.

## Decision It Supports

Whether to introduce an optional history-blend mode in serving, and which parameter region is most promising.

## Primary Metrics

Hit@5, Recall@5, MAP@5, NDCG@5, and MRR (aggregated per-user with recs_004 semantics).

## Setup Notes

- Target/proxy: for each eval user, treat one liked review as query and other liked eval games as positives.
- Feature choice: query text + optional train-history blend from same user.
- Leakage guard: train history excludes query app and eval positives, and excludes train rows after query timestamp.
- Split strategy: eval split controlled by `RECS004_EVAL_SPLIT` (`val` default, `test` optional holdout).


In [1]:
from __future__ import annotations

import json
import math
import os
import time
from pathlib import Path
from itertools import product

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)


In [2]:
def _repo_root() -> Path:
    here = Path.cwd().resolve()
    for d in [here, *here.parents]:
        if (d / 'pyproject.toml').is_file():
            return d
    raise RuntimeError(f'Could not find repo root from cwd={here}')

REPO_ROOT = _repo_root()
PROCESSED = REPO_ROOT / 'data' / 'processed'
VAL_PARQUET = PROCESSED / 'steam_reviews_cleaned_english_val_norm.parquet'
TRAIN_PARQUET = PROCESSED / 'steam_reviews_cleaned_english_train_norm.parquet'
TEST_PARQUET = PROCESSED / 'steam_reviews_cleaned_english_test_norm.parquet'

split = os.environ.get('RECS004_EVAL_SPLIT', 'val').strip().lower()
if split == 'test':
    EVAL_PARQUET = TEST_PARQUET
    EVAL_SPLIT_NAME = 'test'
elif split == 'train':
    EVAL_PARQUET = TRAIN_PARQUET
    EVAL_SPLIT_NAME = 'train'
else:
    EVAL_PARQUET = VAL_PARQUET if VAL_PARQUET.is_file() else TRAIN_PARQUET
    EVAL_SPLIT_NAME = 'val' if EVAL_PARQUET == VAL_PARQUET else 'train'

for p in (EVAL_PARQUET, TRAIN_PARQUET):
    if not p.is_file():
        raise FileNotFoundError(p)

from steam_review_ml.recommender.retrieve import ContentRetriever
retriever = ContentRetriever()
indexed_apps = set(retriever.app_ids.tolist())

print('Eval split:', EVAL_SPLIT_NAME, '|', EVAL_PARQUET.name)
print('Indexed games:', len(indexed_apps))


Eval split: val | steam_reviews_cleaned_english_val_norm.parquet
Indexed games: 315


In [3]:
# Runtime + grid config (requested): medium grid, 5000-user cap
from steam_review_ml.constants import PROJECT_RANDOM_SEED

RNG_SEED = PROJECT_RANDOM_SEED
MAX_USERS = 1000
MIN_REVIEW_CHARS = 30
MAX_HISTORY_ROWS_PER_USER = 200
TOP_K = 5
RETRIEVAL_K = 200  # used for MRR scan depth

ALPHAS = [0.05, 0.10, 0.20, 0.30, 0.40]
MIN_SIMS = [0.0, 0.1, 0.2, 0.3, 0.4]
HISTORY_TOP_KS = [1, 2, 3, 5]
# ALPHAS = [0.05, 0.10,]
# MIN_SIMS = [0.0, 0.1,]
# HISTORY_TOP_KS = [1, ]

GRID = [
    {'history_alpha': a, 'history_min_similarity': s, 'history_top_k': t}
    for a, s, t in product(ALPHAS, MIN_SIMS, HISTORY_TOP_KS)
]

print('Grid size:', len(GRID))


Grid size: 100


In [4]:
def recall_at_k(ranked_app_ids: np.ndarray, positives: set[int], k: int) -> float:
    if not positives:
        return float('nan')
    top = set(int(a) for a in ranked_app_ids[:k])
    return len(top & positives) / len(positives)

def hit_rate_at_k(ranked_app_ids: np.ndarray, positives: set[int], k: int) -> float:
    top = set(int(a) for a in ranked_app_ids[:k])
    return 1.0 if (top & positives) else 0.0

def average_precision_at_k(ranked_app_ids: np.ndarray, positives: set[int], k: int) -> float:
    if not positives:
        return float('nan')
    hits = 0
    prec_sum = 0.0
    for rank, app_id in enumerate(ranked_app_ids[:k].tolist(), start=1):
        if int(app_id) in positives:
            hits += 1
            prec_sum += hits / rank
    return prec_sum / len(positives)

def ndcg_at_k(ranked_app_ids: np.ndarray, positives: set[int], k: int) -> float:
    if not positives:
        return float('nan')
    gains = [1.0 if int(a) in positives else 0.0 for a in ranked_app_ids[:k]]

    def dcg(vals: list[float]) -> float:
        return sum(rel / math.log2(idx + 2) for idx, rel in enumerate(vals))

    ideal_len = min(len(positives), k)
    ideal = [1.0] * ideal_len + [0.0] * max(0, k - ideal_len)
    idcg = dcg(ideal)
    if idcg <= 1e-12:
        return 0.0
    return dcg(gains) / idcg

def mrr(ranked_app_ids: np.ndarray, positives: set[int]) -> float:
    if not positives:
        return float('nan')
    for rank, app_id in enumerate(ranked_app_ids.tolist(), start=1):
        if int(app_id) in positives:
            return 1.0 / rank
    return 0.0

def summarize(metric_lists: dict[str, list[float]]) -> dict[str, float]:
    out = {}
    for metric, vals in metric_lists.items():
        arr = np.asarray(vals, dtype=np.float64)
        if metric.startswith(('recall', 'map', 'ndcg')) or metric == 'mrr':
            out[metric] = float(np.nanmean(arr))
        else:
            out[metric] = float(arr.mean())
    return out


In [5]:
rng = np.random.default_rng(RNG_SEED)
USER_COL = 'author.steamid'
TIME_COL = 'timestamp_created'
usecols = [USER_COL, 'app_id', 'review', 'recommended', TIME_COL]

def _load_split_df(path: Path) -> pd.DataFrame:
    d = pd.read_parquet(path, columns=usecols)
    d = d.loc[d['recommended'] == 1].copy()
    d['review'] = d['review'].fillna('').astype(str)
    d = d[d['review'].str.len() >= MIN_REVIEW_CHARS]
    d = d[d['app_id'].isin(indexed_apps)]
    d['ts'] = pd.to_numeric(d[TIME_COL], errors='coerce')
    d = d.dropna(subset=['ts'])
    d['ts'] = d['ts'].astype(np.float64)
    return d

df_eval = _load_split_df(EVAL_PARQUET)
df_train = _load_split_df(TRAIN_PARQUET)

uc = df_eval.groupby(USER_COL)['app_id'].nunique()
multi = uc[uc >= 2].index
multi = pd.Index(rng.permutation(multi.values)[: min(len(multi), MAX_USERS)])

examples = []
for uid in multi:
    sub_v = df_eval[df_eval[USER_COL] == uid]
    sub_t = df_train[df_train[USER_COL] == uid]
    apps = sub_v['app_id'].unique().tolist()

    q_app = int(rng.choice(apps))
    rows_q = sub_v[sub_v['app_id'] == q_app]
    row_q = rows_q.iloc[int(rng.integers(0, len(rows_q)))]

    positives = {int(a) for a in apps if int(a) != q_app}
    if not positives:
        continue

    query_text = str(row_q['review'])
    query_ts = float(row_q['ts'])
    blocklist = positives | {q_app}

    history_rows = []
    for _, r in sub_t.iterrows():
        aid = int(r['app_id'])
        ts = float(r['ts'])
        if aid in blocklist:
            continue
        if ts > query_ts:
            continue
        history_rows.append(str(r['review']))

    rng.shuffle(history_rows)
    if len(history_rows) > MAX_HISTORY_ROWS_PER_USER:
        history_rows = history_rows[:MAX_HISTORY_ROWS_PER_USER]

    examples.append({
        'steamid': uid,
        'query_app_id': q_app,
        'query_text': query_text,
        'positives': positives,
        'history_texts': history_rows,
    })

print('Examples:', len(examples))
if examples:
    print('History texts per user (median):', int(np.median([len(e['history_texts']) for e in examples])))


Examples: 1000
History texts per user (median): 0


In [6]:
def _fmt_duration(seconds: float) -> str:
    seconds = max(0, int(seconds))
    h, rem = divmod(seconds, 3600)
    m, s = divmod(rem, 60)
    if h > 0:
        return f"{h}h {m:02d}m {s:02d}s"
    return f"{m:02d}m {s:02d}s"


def evaluate_config(
    cfg: dict[str, float | int],
    *,
    progress_every_users: int = 500,
    cfg_idx: int | None = None,
    total_cfgs: int | None = None,
) -> dict[str, float]:
    agg = {'hit@5': [], 'recall@5': [], 'map@5': [], 'ndcg@5': [], 'mrr': []}
    n_examples = len(examples)
    start_cfg = time.perf_counter()

    if cfg_idx is not None and total_cfgs is not None:
        print(
            f"[Config {cfg_idx}/{total_cfgs}] start "
            f"alpha={cfg['history_alpha']}, min_sim={cfg['history_min_similarity']}, top_k={cfg['history_top_k']}"
        )

    for i, ex in enumerate(examples, start=1):
        hits = retriever.top_k(
            ex['query_text'],
            k=RETRIEVAL_K,
            structured=False,
            exclude_app_ids={int(ex['query_app_id'])},
            history_texts=ex['history_texts'],
            history_blend_alpha=float(cfg['history_alpha']),
            history_top_k=int(cfg['history_top_k']),
            history_min_similarity=float(cfg['history_min_similarity']),
        )
        ranked = hits['app_id'].to_numpy(dtype=np.int64)
        pos = ex['positives']

        agg['hit@5'].append(hit_rate_at_k(ranked, pos, TOP_K))
        agg['recall@5'].append(recall_at_k(ranked, pos, TOP_K))
        agg['map@5'].append(average_precision_at_k(ranked, pos, TOP_K))
        agg['ndcg@5'].append(ndcg_at_k(ranked, pos, TOP_K))
        agg['mrr'].append(mrr(ranked, pos))

        if progress_every_users > 0 and (i % progress_every_users == 0 or i == n_examples):
            elapsed = time.perf_counter() - start_cfg
            done_pct = 100.0 * i / n_examples
            rate = elapsed / i
            eta = rate * (n_examples - i)
            print(
                f"    users {i}/{n_examples} ({done_pct:5.1f}%) | "
                f"elapsed {_fmt_duration(elapsed)} | eta {_fmt_duration(eta)}"
            )

    return summarize(agg)


# Baseline (raw query, no history blend)
baseline_cfg = {'history_alpha': 0.0, 'history_min_similarity': 0.2, 'history_top_k': 3}
print('Computing baseline config...')
start_all = time.perf_counter()
baseline_metrics = evaluate_config(baseline_cfg, progress_every_users=1000)
print('Baseline done:', baseline_metrics)

rows = []
total_cfgs = len(GRID)
for idx, cfg in enumerate(GRID, start=1):
    cfg_start = time.perf_counter()
    metrics = evaluate_config(
        cfg,
        progress_every_users=1000,
        cfg_idx=idx,
        total_cfgs=total_cfgs,
    )
    row = {**cfg, **metrics}
    for m in ['hit@5', 'recall@5', 'map@5', 'ndcg@5', 'mrr']:
        row[f'delta_{m}'] = row[m] - baseline_metrics[m]
    rows.append(row)

    cfg_elapsed = time.perf_counter() - cfg_start
    all_elapsed = time.perf_counter() - start_all
    avg_per_cfg = all_elapsed / idx
    remaining = avg_per_cfg * (total_cfgs - idx)
    print(
        f"[Config {idx}/{total_cfgs}] done in {_fmt_duration(cfg_elapsed)} | "
        f"total {_fmt_duration(all_elapsed)} | eta {_fmt_duration(remaining)}"
    )

grid_df = pd.DataFrame(rows).sort_values('map@5', ascending=False).reset_index(drop=True)

print('Baseline (alpha=0):', baseline_metrics)
print('Grid search complete. Total runtime:', _fmt_duration(time.perf_counter() - start_all))
display(grid_df.head(20))


Computing baseline config...


2026-04-21 18:24:05.989839: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776810245.999423  275935 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776810246.002282  275935 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776810246.010159  275935 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776810246.010172  275935 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776810246.010174  275935 computation_placer.cc:177] computation placer alr

    users 1000/1000 (100.0%) | elapsed 00m 21s | eta 00m 00s
Baseline done: {'hit@5': 0.064, 'recall@5': 0.064, 'map@5': 0.03483333333333333, 'ndcg@5': 0.041976874058512165, 'mrr': 0.050136731386834894}
[Config 1/100] start alpha=0.05, min_sim=0.0, top_k=1
    users 1000/1000 (100.0%) | elapsed 00m 10s | eta 00m 00s
[Config 1/100] done in 00m 10s | total 00m 31s | eta 52m 22s
[Config 2/100] start alpha=0.05, min_sim=0.0, top_k=2
    users 1000/1000 (100.0%) | elapsed 00m 10s | eta 00m 00s
[Config 2/100] done in 00m 10s | total 00m 42s | eta 34m 39s
[Config 3/100] start alpha=0.05, min_sim=0.0, top_k=3
    users 1000/1000 (100.0%) | elapsed 00m 12s | eta 00m 00s
[Config 3/100] done in 00m 12s | total 00m 54s | eta 29m 23s
[Config 4/100] start alpha=0.05, min_sim=0.0, top_k=5
    users 1000/1000 (100.0%) | elapsed 00m 11s | eta 00m 00s
[Config 4/100] done in 00m 11s | total 01m 06s | eta 26m 36s
[Config 5/100] start alpha=0.05, min_sim=0.1, top_k=1
    users 1000/1000 (100.0%) | elapsed 

,history_alpha,history_min_similarity,history_top_k,hit@5,recall@5,map@5,ndcg@5,mrr,delta_hit@5,delta_recall@5,delta_map@5,delta_ndcg@5,delta_mrr
0,0.05,0.0,1,0.064,0.064,0.034833,0.041977,0.050137,0.0,0.0,0.0,0.0,0.0
1,0.05,0.0,2,0.064,0.064,0.034833,0.041977,0.050137,0.0,0.0,0.0,0.0,0.0
2,0.05,0.0,3,0.064,0.064,0.034833,0.041977,0.050137,0.0,0.0,0.0,0.0,0.0
3,0.05,0.0,5,0.064,0.064,0.034833,0.041977,0.050137,0.0,0.0,0.0,0.0,0.0
4,0.05,0.1,1,0.064,0.064,0.034833,0.041977,0.050137,0.0,0.0,0.0,0.0,0.0
5,0.05,0.1,2,0.064,0.064,0.034833,0.041977,0.050137,0.0,0.0,0.0,0.0,0.0
6,0.05,0.1,3,0.064,0.064,0.034833,0.041977,0.050137,0.0,0.0,0.0,0.0,0.0
7,0.05,0.1,5,0.064,0.064,0.034833,0.041977,0.050137,0.0,0.0,0.0,0.0,0.0
8,0.05,0.2,1,0.064,0.064,0.034833,0.041977,0.050137,0.0,0.0,0.0,0.0,0.0
9,0.05,0.2,2,0.064,0.064,0.034833,0.041977,0.050137,0.0,0.0,0.0,0.0,0.0


In [7]:
metric_cols = ['hit@5', 'recall@5', 'map@5', 'ndcg@5', 'mrr']
best_per_metric = []
for m in metric_cols:
    r = grid_df.sort_values(m, ascending=False).iloc[0]
    best_per_metric.append({
        'metric': m,
        'best_value': float(r[m]),
        'history_alpha': float(r['history_alpha']),
        'history_min_similarity': float(r['history_min_similarity']),
        'history_top_k': int(r['history_top_k']),
        'delta_vs_baseline': float(r[f'delta_{m}']),
    })

best_df = pd.DataFrame(best_per_metric)
display(best_df)

OUT = REPO_ROOT / 'artifacts' / 'recs' / 'eval_history_blend_gridsearch_at5.csv'
grid_df.to_csv(OUT, index=False)
print('Wrote:', OUT)


,metric,best_value,history_alpha,history_min_similarity,history_top_k,delta_vs_baseline
0,hit@5,0.064000,0.05,0.0,1,0.0
1,recall@5,0.064000,0.05,0.0,1,0.0
2,map@5,0.034833,0.05,0.0,1,0.0
3,ndcg@5,0.041977,0.05,0.0,1,0.0
4,mrr,0.050137,0.05,0.0,1,0.0


Wrote: /home/ryanr/workspace/steam_recommendations/artifacts/recs/eval_history_blend_gridsearch_at5.csv
